In [21]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm

In [33]:
sleep_data = pd.read_csv('dataset_2191_sleep.csv', na_values=['?'])
sleep_data.head()

,body_weight,brain_weight,max_life_span,gestation_time,predation_index,sleep_exposure_index,danger_index,total_sleep
0,6654.000,5712.0,38.6,645.0,3,5,3,3.3
1,1.000,6.6,4.5,42.0,3,1,3,8.3
2,3.385,44.5,14.0,60.0,1,1,1,12.5
3,0.920,5.7,NaN,25.0,5,2,3,16.5
4,2547.000,4603.0,69.0,624.0,3,5,4,3.9


In [34]:
sleep_data.describe()

,body_weight,brain_weight,max_life_span,gestation_time,predation_index,sleep_exposure_index,danger_index,total_sleep
count,62.000000,62.000000,58.000000,58.000000,62.000000,62.000000,62.000000,58.000000
mean,198.789984,283.134194,19.877586,142.353448,2.870968,2.419355,2.612903,10.532759
std,899.158011,930.278942,18.206255,146.805039,1.476414,1.604792,1.441252,4.606760
min,0.005000,0.140000,2.000000,12.000000,1.000000,1.000000,1.000000,2.600000
25%,0.600000,4.250000,6.625000,35.750000,2.000000,1.000000,1.000000,8.050000
50%,3.342500,17.250000,15.100000,79.000000,3.000000,2.000000,2.000000,10.450000
75%,48.202500,166.000000,27.750000,207.500000,4.000000,4.000000,4.000000,13.200000
max,6654.000000,5712.000000,100.000000,645.000000,5.000000,5.000000,5.000000,19.900000


In [35]:
columns = sleep_data.columns
for column in columns:
    sleep_data[column] = pd.to_numeric(sleep_data[column], errors='coerce')

In [36]:
sleep_data = sleep_data.sort_values(by='max_life_span', ascending=True)

px.scatter(sleep_data, x='max_life_span', y='total_sleep', trendline='ols')

In [37]:
columns =columns.drop('total_sleep')
for column in columns:
    fig = px.scatter(
        sleep_data,
        x=column,
        y='total_sleep',
        trendline='ols',
        title=f'Total Sleep vs {column}'
    )
    fig.show()

In [38]:
model1 = sm.OLS.from_formula('total_sleep ~ body_weight + brain_weight + predation_index + sleep_exposure_index + danger_index', data=sleep_data)

In [39]:
model1_results = model1.fit()
print(model1_results.summary())

                            OLS Regression Results                            
Dep. Variable:            total_sleep   R-squared:                       0.546
Model:                            OLS   Adj. R-squared:                  0.502
Method:                 Least Squares   F-statistic:                     12.49
Date:                Wed, 01 Oct 2025   Prob (F-statistic):           5.60e-08
Time:                        19:54:14   Log-Likelihood:                -147.52
No. Observations:                  58   AIC:                             307.0
Df Residuals:                      52   BIC:                             319.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               14.7853 

## Analysis of Model

From this model we can see issues with multicolinearity and other influencing variables. We can also see high p-values for some of the variables. While the model does account for 54% of the variance in total sleep, it does not account for all of it, making it only partially useful.

## Future Work
I plan to eliminate and modify some variables to try and improve the model.

In [40]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [43]:
X = sleep_data.drop(columns=['total_sleep'])

In [45]:
print(X.isna().sum())      # count NaNs per column
print(np.isinf(X).sum())   # count infs per column

body_weight             0
brain_weight            0
max_life_span           4
gestation_time          4
predation_index         0
sleep_exposure_index    0
danger_index            0
dtype: int64
body_weight             0
brain_weight            0
max_life_span           0
gestation_time          0
predation_index         0
sleep_exposure_index    0
danger_index            0
dtype: int64


In [53]:
X = sleep_data[["body_weight", "brain_weight", "max_life_span",
        "gestation_time", "predation_index", 
        "sleep_exposure_index", "danger_index"]]

X = X.replace([np.inf, -np.inf], np.nan)  # just in case
X = X.dropna()  # remove rows with NaN


In [54]:
print(X.isna().sum())      # count NaNs per column
print(np.isinf(X).sum()) 

body_weight             0
brain_weight            0
max_life_span           0
gestation_time          0
predation_index         0
sleep_exposure_index    0
danger_index            0
dtype: int64
body_weight             0
brain_weight            0
max_life_span           0
gestation_time          0
predation_index         0
sleep_exposure_index    0
danger_index            0
dtype: int64


In [55]:


# Add constant for intercept
X = sm.add_constant(X)

# Compute VIF
vif = pd.DataFrame()
vif["Features"] = X.columns
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif)

               Features  VIF Factor
0                 const    7.437607
1           body_weight   12.838008
2          brain_weight   16.444563
3         max_life_span    2.741350
4        gestation_time    3.966097
5       predation_index    9.593595
6  sleep_exposure_index    5.052113
7          danger_index   15.037885


In [56]:
model2 = sm.OLS.from_formula('total_sleep ~ body_weight + max_life_span + gestation_time + predation_index + sleep_exposure_index + danger_index', data=sleep_data)

In [57]:
model2_results = model2.fit()
print(model2_results.summary())

                            OLS Regression Results                            
Dep. Variable:            total_sleep   R-squared:                       0.667
Model:                            OLS   Adj. R-squared:                  0.621
Method:                 Least Squares   F-statistic:                     14.67
Date:                Wed, 01 Oct 2025   Prob (F-statistic):           4.07e-09
Time:                        19:57:50   Log-Likelihood:                -122.55
No. Observations:                  51   AIC:                             259.1
Df Residuals:                      44   BIC:                             272.6
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               16.2284 

## Analysis of Model2

Model 2 looks better than model 1, but it still has room for improvement. It has a higher R-squared value but there still sems to be some multicolinearity issues.

In [58]:
X = sleep_data[["body_weight", "max_life_span",
        "gestation_time", "predation_index", 
        "sleep_exposure_index", "danger_index"]]

X = X.replace([np.inf, -np.inf], np.nan)  # just in case
X = X.dropna()  # remove rows with NaN


In [60]:
vif = pd.DataFrame()
vif["Features"] = X.columns
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif)

               Features  VIF Factor
0           body_weight    1.999496
1         max_life_span    3.392060
2        gestation_time    7.401768
3       predation_index   37.551229
4  sleep_exposure_index   16.163821
5          danger_index   62.992400


In [62]:
model3 = sm.OLS.from_formula('total_sleep ~ body_weight + max_life_span + gestation_time + predation_index + sleep_exposure_index', data=sleep_data)

model3_results = model3.fit()
print(model3_results.summary())

                            OLS Regression Results                            
Dep. Variable:            total_sleep   R-squared:                       0.570
Model:                            OLS   Adj. R-squared:                  0.522
Method:                 Least Squares   F-statistic:                     11.94
Date:                Wed, 01 Oct 2025   Prob (F-statistic):           2.21e-07
Time:                        20:02:47   Log-Likelihood:                -129.04
No. Observations:                  51   AIC:                             270.1
Df Residuals:                      45   BIC:                             281.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               16.9312 

In [61]:
X = sleep_data[["body_weight", "max_life_span", 
        "gestation_time", "predation_index", 
        "sleep_exposure_index"]]
X = X.replace([np.inf, -np.inf], np.nan)  # just in case
X = X.dropna()  # remove rows with NaN

vif = pd.DataFrame()
vif["Features"] = X.columns
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif)

               Features  VIF Factor
0           body_weight    1.952253
1         max_life_span    3.373690
2        gestation_time    7.315892
3       predation_index    6.288741
4  sleep_exposure_index   10.570227


In [63]:
model4 = sm.OLS.from_formula('total_sleep ~ body_weight + max_life_span + gestation_time + predation_index', data=sleep_data)

model4_results = model4.fit()
print(model4_results.summary())

                            OLS Regression Results                            
Dep. Variable:            total_sleep   R-squared:                       0.560
Model:                            OLS   Adj. R-squared:                  0.522
Method:                 Least Squares   F-statistic:                     14.65
Date:                Wed, 01 Oct 2025   Prob (F-statistic):           8.65e-08
Time:                        20:03:16   Log-Likelihood:                -129.62
No. Observations:                  51   AIC:                             269.2
Df Residuals:                      46   BIC:                             278.9
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          17.0771      1.160     

In [64]:
X = sleep_data[["body_weight", "max_life_span", 
        "gestation_time", "predation_index", 
        ]]
X = X.replace([np.inf, -np.inf], np.nan)  # just in case
X = X.dropna()  # remove rows with NaN

vif = pd.DataFrame()
vif["Features"] = X.columns
vif["VIF Factor"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif)

          Features  VIF Factor
0      body_weight    1.950709
1    max_life_span    3.274147
2   gestation_time    6.156475
3  predation_index    2.272155


In [65]:
residuals = model4_results.resid

In [66]:
residuals

37    1.322680
58    3.423321
31   -1.302855
60    5.147174
36    1.525801
14   -6.498822
22    1.446527
1    -4.107019
56    0.027595
47    0.367905
38    5.155135
16   -2.905542
51   -3.222833
39    5.402843
50   -3.231227
10    4.056231
57   -3.698293
27   -1.131731
26    3.682505
48   -4.747228
17   -1.201866
41    5.545865
54   -1.462920
43   -0.252056
46   -0.494705
2    -1.782725
25   -1.194195
52   -4.838186
45   -1.443793
6     5.026413
21   -3.617667
53   -3.577545
42    2.717157
29   -2.537289
59    3.031778
32    5.652763
44   -0.887963
5     1.983414
8     0.611805
49   -0.962782
11   -0.889629
7     2.694450
0    -1.042763
23    2.097735
13    0.482997
24   -2.379534
28   -0.424849
15   -4.247655
9    -0.279840
4     3.001295
33   -0.039876
dtype: float64

In [69]:
resid_df = pd.DataFrame({
    "index": residuals.index,
    "residuals": residuals
})

fig = px.scatter(resid_df, x="index", y="residuals",
                 title="Residuals Plot",
                 labels={"index": "Observation", "residuals": "Residuals"},
                 trendline="ols")
fig.show()

In [73]:
fitted = model4_results.fittedvalues
residuals = model4_results.resid

# Fit a quadratic trend to residuals
coeffs = np.polyfit(fitted, residuals, 2)  # degree 2
quad_fit = np.polyval(coeffs, fitted)

resid_df = pd.DataFrame({
    "fitted": fitted,
    "residuals": residuals,
    "quad_fit": quad_fit
})

fig = px.scatter(resid_df, x="fitted", y="residuals",
                 title="Residuals vs Fitted with Quadratic Trend",
                 labels={"fitted": "Fitted Values", "residuals": "Residuals"})
fig.add_traces(px.line(resid_df, x="fitted", y="quad_fit").data)  # add quadratic trend
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()